<a href="https://colab.research.google.com/github/JonathanJulDiaz/devfest-demo/blob/main/credit_risk_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/JonathanJulDiaz/devfest-demo/refs/heads/main/credit_risk_dataset.csv"
df = pd.read_csv(url)

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df['person_emp_length']= df['person_emp_length'].fillna(df['person_emp_length'].median())
df['loan_int_rate'] = df['loan_int_rate'].fillna(df['loan_int_rate'].mean())

df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df = df[df['person_age'] < 90]
df = df[df['person_age'] - df['person_emp_length'] > 18]

df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='loan_status', hue='loan_status', data=df, palette=['blue', 'red'], legend=False)
plt.title("Distribución del estado del préstamo")
plt.xlabel("Estado (0 = sin riesgo, 1 = con riesgo)")
plt.ylabel("Cantidad")
plt.show()

In [ ]:
orden = df[df['loan_status'] == 0]['loan_intent'].value_counts().index

sns.countplot(y='loan_intent', hue='loan_status', data=df, order=orden)
plt.title("Tipo de préstamo vs Estado")
plt.legend(title="Riesgo")
plt.show()

In [ ]:
df_procesado = pd.get_dummies(df, drop_first=True)

In [ ]:
corr_var = df_procesado.corr()['loan_status'].sort_values(ascending=False)
corr_var = corr_var.drop('loan_status')

plt.figure(figsize=(8,6))
sns.barplot(x=corr_var, hue=corr_var.index, y=corr_var.index, palette="coolwarm")
plt.show()

In [ ]:
corr_var.head()

In [ ]:
from sklearn.model_selection import train_test_split

X = df_procesado.drop(columns=['loan_status'])
y = df_procesado['loan_status']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=25)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=25)
model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_pred = model.predict(X_test)

print(f"Precisión: {accuracy_score(y_test, y_pred)}")

print("\nReporte de clasificación:")
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("Matriz de confusión")
plt.xlabel("Predicción")
plt.ylabel("Real")
plt.show()

In [ ]:
relevancia_variables = pd.Series(model.feature_importances_, index=X_train.columns)
relevancia_variables = relevancia_variables.sort_values()
relevancia_variables.tail(10).plot(kind='barh', figsize=(8,6), color='orange')
plt.title("Importancia de las variables según RandomForest")
plt.show()

In [ ]:
relevancia_variables.sort_values(ascending=False)